# Contrato lógico de ingestão BigQuery

## Escopo e limite de evidência

Este catálogo documenta os campos lógicos que uma futura ingestão precisa
receber. Ele **não** consulta BigQuery e não supõe projeto, dataset, tabela,
coluna, partição, watermark ou regra de atualização. Tudo que depende do schema
físico permanece como `não fornecido` até validação da Globo.

As fontes ServiceNow e NPAW/Youbora são domínios documentados via BigQuery. Os
seis campos de `frontend_events_sample` usam
`source_system="amostra frontend; origem BigQuery não comprovada"`: vêm de
exemplos de frontend e ficam separados do contrato confirmado.

In [6]:
from dump_explorer.contract import (
    FRONTEND_SAMPLE_SOURCE,
    OUTPUTS_DIR,
    UNKNOWN_PHYSICAL_MAPPING,
    build_logical_contract,
    export_contract_csv,
    validate_physical_mapping,
)

contract = build_logical_contract()
physical_columns = [
    "bigquery_table",
    "bigquery_column",
    "bigquery_partition_column",
    "bigquery_write_watermark",
    "update_semantics",
]

## 1. Campos lógicos de domínios documentados via BigQuery

NPAW tem atraso estrutural de aproximadamente 28 horas. Em ServiceNow, `opened_at` é ticket-open time, não o início do incidente.

In [7]:
confirmed_domains = contract.loc[
    contract["source_system"] != FRONTEND_SAMPLE_SOURCE
]
display(confirmed_domains)

domain_summary = confirmed_domains.groupby(
    ["source_system", "logical_table", "grain", "known_delay"], as_index=False
).size()
display(domain_summary)

,logical_table,logical_field,source_system,grain,logical_type,key_role,event_timestamp,watermark_candidate,known_delay,storage_recommendation,sensitivity,bigquery_table,bigquery_column,bigquery_partition_column,bigquery_write_watermark,update_semantics
0,incidents,opened_at,ServiceNow; domínio documentado via BigQuery,ticket,"timestamp (epoch_ms UTC; ticket-open time, not...",event timestamp,True,True,ticket-open typically +5-40min after incident ...,hot+cold,operational,não fornecido,não fornecido,não fornecido,não fornecido,não fornecido
1,incidents,sys_id,ServiceNow; domínio documentado via BigQuery,ticket,string,natural key,False,False,não fornecido,hot+cold,operational,não fornecido,não fornecido,não fornecido,não fornecido,não fornecido
2,incidents,number,ServiceNow; domínio documentado via BigQuery,ticket,string,attribute,False,False,não fornecido,hot+cold,operational,não fornecido,não fornecido,não fornecido,não fornecido,não fornecido
3,incidents,resolved_at,ServiceNow; domínio documentado via BigQuery,ticket,timestamp (epoch_ms UTC),attribute,True,True,não fornecido,hot+cold,operational,não fornecido,não fornecido,não fornecido,não fornecido,não fornecido
4,incidents,closed_at,ServiceNow; domínio documentado via BigQuery,ticket,timestamp (epoch_ms UTC),attribute,True,True,não fornecido,hot+cold,operational,não fornecido,não fornecido,não fornecido,não fornecido,não fornecido
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,error_cat_hourly,hour_ts,NPAW/Youbora; domínio documentado via BigQuery,hora por kind por categoria,timestamp (epoch_ms UTC),natural key component,True,True,~28h,hot+cold,aggregate,não fornecido,não fornecido,não fornecido,não fornecido,não fornecido
77,error_cat_hourly,kind,NPAW/Youbora; domínio documentado via BigQuery,hora por kind por categoria,string,natural key component,False,False,~28h,hot+cold,aggregate,não fornecido,não fornecido,não fornecido,não fornecido,não fornecido
78,error_cat_hourly,category,NPAW/Youbora; domínio documentado via BigQuery,hora por kind por categoria,string,natural key component,False,False,~28h,hot+cold,aggregate,não fornecido,não fornecido,não fornecido,não fornecido,não fornecido
79,error_cat_hourly,errors,NPAW/Youbora; domínio documentado via BigQuery,hora por kind por categoria,integer,measure,False,False,~28h,hot+cold,aggregate,não fornecido,não fornecido,não fornecido,não fornecido,não fornecido


,source_system,logical_table,grain,known_delay,size
0,NPAW/Youbora; domínio documentado via BigQuery,device_hourly,hora por dispositivo,~28h,4
1,NPAW/Youbora; domínio documentado via BigQuery,error_cat_hourly,hora por kind por categoria,~28h,5
2,NPAW/Youbora; domínio documentado via BigQuery,geo_hourly,hora por UF por operadora,~28h,5
3,NPAW/Youbora; domínio documentado via BigQuery,live_active_15min,15 min por sinal,~28h,10
4,NPAW/Youbora; domínio documentado via BigQuery,live_error_attr_hourly,hora por sinal por CDN por camada por HTTP por...,~28h,7
5,NPAW/Youbora; domínio documentado via BigQuery,live_qoe_hourly,hora por sinal,~28h,12
6,NPAW/Youbora; domínio documentado via BigQuery,player_hourly,hora por player,~28h,4
7,NPAW/Youbora; domínio documentado via BigQuery,player_live_hourly,hora por player,~28h,10
8,ServiceNow; domínio documentado via BigQuery,incidents,ticket,não fornecido,23
9,ServiceNow; domínio documentado via BigQuery,incidents,ticket,ticket-open typically +5-40min after incident ...,1


## 2. Amostras de eventos de frontend

A seção abaixo não é um mapeamento BigQuery e não autoriza ingestão de identificadores individuais. Ela registra apenas os campos lógicos selecionados dos seis exemplos fornecidos.

In [8]:
frontend_samples = contract.loc[
    contract["source_system"] == FRONTEND_SAMPLE_SOURCE
]
display(frontend_samples)

,logical_table,logical_field,source_system,grain,logical_type,key_role,event_timestamp,watermark_candidate,known_delay,storage_recommendation,sensitivity,bigquery_table,bigquery_column,bigquery_partition_column,bigquery_write_watermark,update_semantics
81,frontend_events_sample,timestamp,amostra frontend; origem BigQuery não comprovada,evento individual de amostra,timestamp,event timestamp,True,True,não fornecido,cold,restricted,não fornecido,não fornecido,não fornecido,não fornecido,não fornecido
82,frontend_events_sample,event_name,amostra frontend; origem BigQuery não comprovada,evento individual de amostra,string,attribute,False,False,não fornecido,cold,restricted,não fornecido,não fornecido,não fornecido,não fornecido,não fornecido
83,frontend_events_sample,app_bucket,amostra frontend; origem BigQuery não comprovada,evento individual de amostra,string,attribute,False,False,não fornecido,cold,restricted,não fornecido,não fornecido,não fornecido,não fornecido,não fornecido
84,frontend_events_sample,app_env,amostra frontend; origem BigQuery não comprovada,evento individual de amostra,string,attribute,False,False,não fornecido,cold,restricted,não fornecido,não fornecido,não fornecido,não fornecido,não fornecido
85,frontend_events_sample,app_target,amostra frontend; origem BigQuery não comprovada,evento individual de amostra,string,attribute,False,False,não fornecido,cold,restricted,não fornecido,não fornecido,não fornecido,não fornecido,não fornecido
86,frontend_events_sample,route_path,amostra frontend; origem BigQuery não comprovada,evento individual de amostra,string,attribute,False,False,não fornecido,cold,restricted,não fornecido,não fornecido,não fornecido,não fornecido,não fornecido


## 3. Lacunas do mapeamento físico

A validação não converte nomes do dump MySQL em nomes físicos BigQuery. Ela torna explícitas as lacunas que bloqueiam uma ingestão incremental segura e idempotente.

In [9]:
try:
    unresolved_physical_fields = contract.loc[
        contract[physical_columns].eq(UNKNOWN_PHYSICAL_MAPPING).any(axis=1)
    ]
    issues = validate_physical_mapping(contract)
    display(unresolved_physical_fields)
    display([issue.__dict__ for issue in issues])
    
    if len(issues) > 0:
        print(f"⚠️  Encontrados {len(issues)} problemas de mapeamento físico que bloqueiam ingestão.")
    else:
        print("✅ Nenhum problema de mapeamento físico encontrado.")
except Exception as e:
    print(f"❌ Erro ao validar mapeamento físico: {e}")

,logical_table,logical_field,source_system,grain,logical_type,key_role,event_timestamp,watermark_candidate,known_delay,storage_recommendation,sensitivity,bigquery_table,bigquery_column,bigquery_partition_column,bigquery_write_watermark,update_semantics
0,incidents,opened_at,ServiceNow; domínio documentado via BigQuery,ticket,"timestamp (epoch_ms UTC; ticket-open time, not...",event timestamp,True,True,ticket-open typically +5-40min after incident ...,hot+cold,operational,não fornecido,não fornecido,não fornecido,não fornecido,não fornecido
1,incidents,sys_id,ServiceNow; domínio documentado via BigQuery,ticket,string,natural key,False,False,não fornecido,hot+cold,operational,não fornecido,não fornecido,não fornecido,não fornecido,não fornecido
2,incidents,number,ServiceNow; domínio documentado via BigQuery,ticket,string,attribute,False,False,não fornecido,hot+cold,operational,não fornecido,não fornecido,não fornecido,não fornecido,não fornecido
3,incidents,resolved_at,ServiceNow; domínio documentado via BigQuery,ticket,timestamp (epoch_ms UTC),attribute,True,True,não fornecido,hot+cold,operational,não fornecido,não fornecido,não fornecido,não fornecido,não fornecido
4,incidents,closed_at,ServiceNow; domínio documentado via BigQuery,ticket,timestamp (epoch_ms UTC),attribute,True,True,não fornecido,hot+cold,operational,não fornecido,não fornecido,não fornecido,não fornecido,não fornecido
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82,frontend_events_sample,event_name,amostra frontend; origem BigQuery não comprovada,evento individual de amostra,string,attribute,False,False,não fornecido,cold,restricted,não fornecido,não fornecido,não fornecido,não fornecido,não fornecido
83,frontend_events_sample,app_bucket,amostra frontend; origem BigQuery não comprovada,evento individual de amostra,string,attribute,False,False,não fornecido,cold,restricted,não fornecido,não fornecido,não fornecido,não fornecido,não fornecido
84,frontend_events_sample,app_env,amostra frontend; origem BigQuery não comprovada,evento individual de amostra,string,attribute,False,False,não fornecido,cold,restricted,não fornecido,não fornecido,não fornecido,não fornecido,não fornecido
85,frontend_events_sample,app_target,amostra frontend; origem BigQuery não comprovada,evento individual de amostra,string,attribute,False,False,não fornecido,cold,restricted,não fornecido,não fornecido,não fornecido,não fornecido,não fornecido


[{'logical_table': 'device_hourly',
  'logical_field': '*',
  'code': 'missing_bigquery_table',
  'message': 'Tabela física BigQuery não fornecida.'},
 {'logical_table': 'device_hourly',
  'logical_field': '*',
  'code': 'missing_partition_column',
  'message': 'Coluna de partição BigQuery não fornecida.'},
 {'logical_table': 'device_hourly',
  'logical_field': '*',
  'code': 'missing_write_watermark',
  'message': 'Watermark de escrita BigQuery não fornecido.'},
 {'logical_table': 'device_hourly',
  'logical_field': '*',
  'code': 'unknown_update_semantics',
  'message': 'Semântica de atualizações, correções e deletes não fornecida.'},
 {'logical_table': 'error_cat_hourly',
  'logical_field': '*',
  'code': 'missing_bigquery_table',
  'message': 'Tabela física BigQuery não fornecida.'},
 {'logical_table': 'error_cat_hourly',
  'logical_field': '*',
  'code': 'missing_partition_column',
  'message': 'Coluna de partição BigQuery não fornecida.'},
 {'logical_table': 'error_cat_hourly',
 

⚠️  Encontrados 127 problemas de mapeamento físico que bloqueiam ingestão.


## 4. Exportação explícita

Nenhum CSV é criado ao abrir ou executar este notebook. Para exportar, altere conscientemente a flag abaixo; o destino é restringido a `analysis/outputs`.

In [10]:
export_csv = False

if export_csv:
    try:
        output_path = OUTPUTS_DIR / "contrato_logico_ingestao.csv"
        exported_path = export_contract_csv(contract, output_path)
        print(f"✅ Contrato exportado para {exported_path}")
    except Exception as e:
        print(f"❌ Erro ao exportar contrato: {e}")
else:
    print("ℹ️  Para exportar o contrato, altere export_csv = True e execute esta célula novamente.")

ℹ️  Para exportar o contrato, altere export_csv = True e execute esta célula novamente.


## Perguntas exatas para enviar à Globo

1. Qual é o **project/dataset/table** físico para cada domínio?
2. Qual é a **partition column** de cada tabela?
3. Qual é o **write watermark** confiável?
4. Qual é a **natural key** de idempotência?
5. Como são tratados **late arrivals**?
6. Como são tratados **corrections/deletes**?
7. Como a **schema evolution** é comunicada e compatibilizada?
8. Quais campos têm **PII** e quais regras de sanitização se aplicam?
9. Quais são as **query quotas** e limites de custo/leitura?